## 6. Conclusioni operative

In [ ]:
##%% md
# Embedding Model Comparison

Questo notebook confronta tre modelli embeddings sullo stesso processed sample:

- `BAAI/bge-small-en-v1.5`
- `sentence-transformers/all-MiniLM-L6-v2`
- `intfloat/e5-base-v2`

Il confronto e' diagnostico: valuta copertura, dimensioni vettoriali, chunking, norme e similarita' locali prima della fase di clustering.
##%% md
## 1. Setup
##%%
from pathlib import Path
import json
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.embedding_pipeline import (
    EmbeddingConfig,
    MODEL_REGISTRY,
    build_embedding_jobs,
    config_for_model,
    get_comparison_model_names,
    get_model_slug,
    run_embedding_pipeline,
)

ENV_PATH = PROJECT_ROOT / ".env"
ENV = dotenv_values(ENV_PATH)

BASE_CONFIG = EmbeddingConfig.from_env(ENV_PATH)
MODEL_NAMES = get_comparison_model_names(ENV_PATH)
MODEL_CONFIGS = [config_for_model(BASE_CONFIG, model_name, output_by_model=True) for model_name in MODEL_NAMES]

FIGURES_ROOT = Path(ENV.get("FIGURES_PATH", "reports/figures/"))
if not FIGURES_ROOT.is_absolute():
    FIGURES_ROOT = PROJECT_ROOT / FIGURES_ROOT
FIGURES_PATH = FIGURES_ROOT / ENV.get("EMBEDDING_COMPARISON_NOTEBOOK_FIGURES_SUBDIR", "embedding_model_comparison")
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

plt.style.use("default")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

print("Modelli configurati:")
for config in MODEL_CONFIGS:
    print(f"- {config.model_name} -> {config.embeddings_output_path.name}")
##%% md
## 2. Input processed e configurazioni
##%%
processed = pd.read_parquet(BASE_CONFIG.processed_input_path)

config_rows = []
for config in MODEL_CONFIGS:
    registry = MODEL_REGISTRY[config.model_name]
    chunk_texts, planned_index = build_embedding_jobs(processed, config)
    config_rows.append(
        {
            "model_name": config.model_name,
            "model_slug": get_model_slug(config.model_name),
            "expected_dimensions": registry["embedding_dimensions"],
            "max_sequence_length": registry["max_sequence_length"],
            "input_prefix": registry["input_prefix"],
            "row_count": len(processed),
            "planned_chunks": len(chunk_texts),
            "mean_chunks_per_email": planned_index["chunk_count"].mean(),
            "max_chunks_per_email": planned_index["chunk_count"].max(),
            "embeddings_output": str(config.embeddings_output_path.relative_to(PROJECT_ROOT)),
            "metadata_output": str(config.embedding_metadata_output_path.relative_to(PROJECT_ROOT)),
        }
    )

config_summary = pd.DataFrame(config_rows)
print(config_summary.to_string(index=False))
##%% md
## 3. Generazione artefatti embeddings

La cella seguente genera solo gli artefatti mancanti. Se un file `.npy`, index e metadata esistono gia', il modello viene saltato per evitare lavoro e download non necessari.
##%%
generation_rows = []
for config in MODEL_CONFIGS:
    artifacts_exist = (
        config.embeddings_output_path.exists()
        and config.embedding_index_output_path.exists()
        and config.embedding_metadata_output_path.exists()
    )
    if artifacts_exist:
        metadata = json.loads(config.embedding_metadata_output_path.read_text(encoding="utf-8"))
        metadata["generation_status"] = "reused_existing_artifacts"
        generation_rows.append(metadata)
        continue

    start = time.perf_counter()
    metadata = run_embedding_pipeline(env_file=ENV_PATH, config=config)
    metadata["generation_seconds"] = round(time.perf_counter() - start, 3)
    metadata["generation_status"] = "generated"
    generation_rows.append(metadata)

generation_summary = pd.DataFrame(generation_rows)
print(generation_summary[[
    "model_name",
    "generation_status",
    "row_count",
    "embedding_dimensions",
    "chunk_count",
    "batch_size",
]].to_string(index=False))
##%% md
## 4. Analisi singolo modello
##%%
analysis_rows = []
per_model_details = {}

for config in MODEL_CONFIGS:
    embeddings = np.load(config.embeddings_output_path)
    index = pd.read_parquet(config.embedding_index_output_path)
    metadata = json.loads(config.embedding_metadata_output_path.read_text(encoding="utf-8"))
    norms = np.linalg.norm(embeddings, axis=1) if embeddings.size else np.array([])
    chunk_counts = index["chunk_count"] if not index.empty else pd.Series(dtype="int64")

    analysis_rows.append(
        {
            "model_name": config.model_name,
            "model_slug": get_model_slug(config.model_name),
            "row_count": int(embeddings.shape[0]),
            "embedding_dimensions": int(embeddings.shape[1]) if embeddings.size else 0,
            "chunk_count": int(metadata.get("chunk_count", 0)),
            "mean_norm": float(norms.mean()) if norms.size else 0.0,
            "std_norm": float(norms.std()) if norms.size else 0.0,
            "min_norm": float(norms.min()) if norms.size else 0.0,
            "max_norm": float(norms.max()) if norms.size else 0.0,
            "mean_chunks_per_email": float(chunk_counts.mean()) if len(chunk_counts) else 0.0,
            "max_chunks_per_email": int(chunk_counts.max()) if len(chunk_counts) else 0,
        }
    )
    per_model_details[config.model_name] = {
        "embeddings": embeddings,
        "index": index,
        "metadata": metadata,
        "norms": norms,
        "chunk_counts": chunk_counts,
    }

analysis_summary = pd.DataFrame(analysis_rows)
print(analysis_summary.to_string(index=False))
##%%
single_fig, axes = plt.subplots(len(MODEL_CONFIGS), 2, figsize=(13, 4 * len(MODEL_CONFIGS)))
if len(MODEL_CONFIGS) == 1:
    axes = np.array([axes])

for row_idx, config in enumerate(MODEL_CONFIGS):
    details = per_model_details[config.model_name]
    slug = get_model_slug(config.model_name)

    norms = details["norms"]
    if norms.size and np.isclose(norms.min(), norms.max()):
        axes[row_idx, 0].bar([f"{norms.mean():.4f}"], [len(norms)])
    else:
        axes[row_idx, 0].hist(norms, bins=30)
    axes[row_idx, 0].set_title(f"{slug}: norme L2")
    axes[row_idx, 0].set_xlabel("norma")
    axes[row_idx, 0].set_ylabel("email")

    chunk_distribution = details["chunk_counts"].value_counts().sort_index()
    axes[row_idx, 1].bar(chunk_distribution.index.astype(str), chunk_distribution.values)
    axes[row_idx, 1].set_title(f"{slug}: chunk per email")
    axes[row_idx, 1].set_xlabel("chunk_count")
    axes[row_idx, 1].set_ylabel("email")

single_fig.tight_layout()
single_fig.savefig(FIGURES_PATH / "embedding_single_model_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()
##%% md
## 5. Confronto tra modelli
##%%
comparison_fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].bar(analysis_summary["model_slug"], analysis_summary["embedding_dimensions"])
axes[0].set_title("Dimensioni embedding")
axes[0].set_ylabel("dimensioni")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(analysis_summary["model_slug"], analysis_summary["chunk_count"])
axes[1].set_title("Chunk totali")
axes[1].set_ylabel("chunk")
axes[1].tick_params(axis="x", rotation=20)

axes[2].bar(analysis_summary["model_slug"], analysis_summary["mean_norm"])
axes[2].set_title("Norma media")
axes[2].set_ylabel("norma")
axes[2].tick_params(axis="x", rotation=20)

comparison_fig.tight_layout()
comparison_fig.savefig(FIGURES_PATH / "embedding_model_comparison_summary.png", dpi=150, bbox_inches="tight")
plt.show()
##%%
sample_size = min(25, len(processed))
sample_positions = np.linspace(0, len(processed) - 1, sample_size, dtype=int)
similarity_rows = []

for config in MODEL_CONFIGS:
    details = per_model_details[config.model_name]
    vectors = details["embeddings"][sample_positions]
    cosine = vectors @ vectors.T
    upper = cosine[np.triu_indices_from(cosine, k=1)]
    similarity_rows.append(
        {
            "model_name": config.model_name,
            "model_slug": get_model_slug(config.model_name),
            "sample_size": sample_size,
            "mean_pairwise_cosine": float(upper.mean()) if upper.size else 0.0,
            "std_pairwise_cosine": float(upper.std()) if upper.size else 0.0,
            "min_pairwise_cosine": float(upper.min()) if upper.size else 0.0,
            "max_pairwise_cosine": float(upper.max()) if upper.size else 0.0,
        }
    )

similarity_summary = pd.DataFrame(similarity_rows)
print(similarity_summary.to_string(index=False))
##%%
comparison_payload = {
    "config_summary": config_summary.to_dict(orient="records"),
    "analysis_summary": analysis_summary.to_dict(orient="records"),
    "similarity_summary": similarity_summary.to_dict(orient="records"),
}
comparison_json_path = FIGURES_PATH / "embedding_model_comparison_summary.json"
comparison_json_path.write_text(json.dumps(comparison_payload, indent=2), encoding="utf-8")
print(f"Figure salvate in: {FIGURES_PATH}")
print(f"Dati confronto salvati in: {comparison_json_path}")
##%% md
## 6. Conclusioni operative

Questo notebook non sceglie automaticamente il modello finale. Produce invece evidenze confrontabili per discutere il candidato da usare nella fase clustering: costo proxy, dimensioni, copertura, chunking, norme e similarita' diagnostiche.

Restored first cell from copilotDiffState originalContent